In [1]:
# Prevent Colab from disconnecting due to inactivity
import IPython
from google.colab import output

def keep_alive():
    IPython.display.display(IPython.display.Javascript('''
        function clickConnect() {
            var buttons = document.querySelectorAll("colab-connect-button");
            if (buttons.length > 0) buttons[0].click();
            setTimeout(clickConnect, 60000);
        }
        clickConnect();
    '''))

keep_alive()

<IPython.core.display.Javascript object>

# 🎙️ Whisper Tiny Fine-Tuning on Your Own Voice

This notebook fine-tunes OpenAI's Whisper Tiny model on your own voice recordings
and measures whether fine-tuning actually improved performance vs baseline Whisper.

### Workflow:
1. Check GPU
2. Install dependencies
3. Upload your audio + transcripts
4. Convert M4A to WAV automatically
5. Validate data
6. Preprocess and build dataset
7. Text normalizer + data collator + metrics
8. Configure experiments
9. Run fine-tuning
10. Compare experiment results
11. Baseline vs fine-tuned WER comparison
12. Download best model

---
**Before running:** Make sure GPU is enabled → Runtime > Change runtime type > T4 GPU

**Text normalization:** Both Whisper output and reference transcripts are normalized
(lowercase, strip punctuation) before WER is computed — so scores are fair and
not inflated by formatting differences.

## Step 0 — Check GPU

In [2]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU found. Go to Runtime > Change runtime type > T4 GPU')

Thu Mar  5 02:49:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1 — Install Dependencies

In [3]:
!apt-get install -q ffmpeg
!pip install -q transformers datasets librosa soundfile jiwer accelerate evaluate scikit-learn
print('✅ Dependencies installed')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.5 MB/s eta 0:00:00
✅ Dependencies installed


## Step 2 — Upload Your Data

### Expected folder structure inside the zip:
```
my_voice_data/
├── clips/
│   ├── clip_001.m4a   ← M4A or WAV both fine
│   ├── clip_002.m4a
│   └── ...
└── metadata.csv
```

### metadata.csv format:
```
file_name,transcript
clip_001.m4a,the weather is nice today
clip_002.m4a,my name is john and I live in seattle
```

Upload the entire `my_voice_data` folder as a zip file below.

In [4]:
from google.colab import files
import zipfile, os

print('📁 Upload your data zip file (my_voice_data.zip)...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

print('✅ Files extracted')
!find /content/my_voice_data -type f | head -20

📁 Upload your data zip file (my_voice_data.zip)...


Saving dataset.zip to dataset.zip
✅ Files extracted
find: ‘/content/my_voice_data’: No such file or directory


## Step 3 — Convert M4A (and other formats) to WAV

The `datasets` library requires WAV internally. This step converts everything automatically and updates metadata.csv to point to the new WAV files. WAV files are skipped.

In [6]:
import subprocess
import os
import pandas as pd

DATA_DIR = '/content/dataset'
CLIPS_DIR = os.path.join(DATA_DIR, 'clips')
METADATA_PATH = os.path.join(DATA_DIR, 'metadata.csv')

df = pd.read_csv(METADATA_PATH)
updated_rows = []

for _, row in df.iterrows():
    fname = row['file_name']
    ext = os.path.splitext(fname)[1].lower()
    in_path = os.path.join(CLIPS_DIR, fname)

    if ext in ('.m4a', '.mp3', '.aac', '.flac'):
        wav_fname = fname.replace(ext, '.wav')
        out_path = os.path.join(CLIPS_DIR, wav_fname)
        result = subprocess.run(
            ['ffmpeg', '-y', '-i', in_path, '-ar', '16000', '-ac', '1', out_path],
            capture_output=True
        )
        if result.returncode == 0:
            print(f'✅ Converted: {fname} → {wav_fname}')
            updated_rows.append({'file_name': wav_fname, 'transcript': row['transcript']})
        else:
            print(f'❌ Failed to convert: {fname}')
            print(result.stderr.decode())
    else:
        print(f'✔️  Already WAV: {fname}')
        updated_rows.append({'file_name': fname, 'transcript': row['transcript']})

# Overwrite metadata.csv with wav filenames
df = pd.DataFrame(updated_rows)
df.to_csv(METADATA_PATH, index=False)

print(f'\n✅ Done. {len(df)} clips ready as WAV.')
print(df.head())

✔️  Already WAV: clip_0001.wav
✔️  Already WAV: clip_0002.wav
✔️  Already WAV: clip_0003.wav
✔️  Already WAV: clip_0004.wav
✔️  Already WAV: clip_0005.wav
✔️  Already WAV: clip_0006.wav
✔️  Already WAV: clip_0007.wav
✔️  Already WAV: clip_0008.wav
✔️  Already WAV: clip_0009.wav
✔️  Already WAV: clip_0010.wav
✔️  Already WAV: clip_0011.wav
✔️  Already WAV: clip_0012.wav
✔️  Already WAV: clip_0013.wav
✔️  Already WAV: clip_0014.wav
✔️  Already WAV: clip_0015.wav
✔️  Already WAV: clip_0016.wav
✔️  Already WAV: clip_0017.wav
✔️  Already WAV: clip_0018.wav
✔️  Already WAV: clip_0019.wav
✔️  Already WAV: clip_0020.wav
✔️  Already WAV: clip_0021.wav
✔️  Already WAV: clip_0022.wav
✔️  Already WAV: clip_0023.wav
✔️  Already WAV: clip_0024.wav
✔️  Already WAV: clip_0025.wav
✔️  Already WAV: clip_0026.wav
✔️  Already WAV: clip_0027.wav
✔️  Already WAV: clip_0028.wav
✔️  Already WAV: clip_0029.wav
✔️  Already WAV: clip_0030.wav
✔️  Already WAV: clip_0031.wav
✔️  Already WAV: clip_0032.wav
✔️  Alre

## Step 4 — Validate Data

In [7]:
import librosa

df = pd.read_csv(METADATA_PATH)
print(f'📊 Total clips: {len(df)}')
print(df.head())

total_duration = 0
issues = []

for _, row in df.iterrows():
    fname = row['file_name']
    path = os.path.join(CLIPS_DIR, fname)
    if not os.path.exists(path):
        issues.append(f'Missing: {fname}')
        continue
    try:
        duration = librosa.get_duration(path=path)
        total_duration += duration
        if duration > 30:
            issues.append(f'Too long ({duration:.1f}s): {fname}')
    except Exception as e:
        issues.append(f'Could not read {fname}: {e}')

print(f'\n⏱️ Total audio duration: {total_duration/60:.1f} minutes')
if issues:
    print('\n⚠️ Issues found:')
    for i in issues: print(f'  - {i}')
else:
    print('✅ All files look good!')

📊 Total clips: 335
       file_name                                         transcript
0  clip_0001.wav                  alright guys let's get it started
1  clip_0002.wav  today we are officially entering the world of ...
2  clip_0003.wav  youve probably heard this term everywhere in j...
3  clip_0004.wav  so today my goal is not to make this scary or ...
4  clip_0005.wav       instead i want you to walk out understanding

⏱️ Total audio duration: 44.1 minutes
✅ All files look good!


## Step 5 — Build Dataset from File Paths

In [9]:
from datasets import Dataset, Audio
from sklearn.model_selection import train_test_split

TARGET_SR = 16000

# Build full file paths
df['path'] = df['file_name'].apply(lambda f: os.path.join(CLIPS_DIR, f))

# Drop any rows with missing or invalid path/transcript
df = df.dropna(subset=['path', 'transcript'])
df = df[df['path'].apply(lambda x: isinstance(x, str))]
df = df[df['transcript'].apply(lambda x: isinstance(x, str) and x.strip() != '')]
df = df.reset_index(drop=True)

# Check all files exist
missing = df[~df['path'].apply(os.path.exists)]
if len(missing) > 0:
    print(f'⚠️ Missing files:')
    for p in missing['path']: print(f'  - {p}')
else:
    print(f'✅ All {len(df)} files found')

# Train/val split (85/15)
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# Build datasets directly from file paths
def make_dataset(split_df):
    return Dataset.from_dict({
        'audio':    split_df['path'].tolist(),
        'sentence': split_df['transcript'].tolist()
    }).cast_column('audio', Audio(sampling_rate=TARGET_SR))

train_dataset = make_dataset(train_df)
val_dataset   = make_dataset(val_df)

print(f'✅ Train clips: {len(train_dataset)}')
print(f'✅ Val clips:   {len(val_dataset)}')
print(f'\n🔍 Sample entry: {train_dataset[0]}')

✅ All 332 files found
✅ Train clips: 282
✅ Val clips:   50

🔍 Sample entry: {'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7dbfcb9670b0>, 'sentence': "it's start very similar similarly to linear regression it takes input features and combines them using weights but"}


## Step 6 — Load Whisper Processor and Prepare Features

In [10]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained('openai/whisper-tiny.en')

def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_features[0]
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids
    return batch

print('🔄 Preparing features...')
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names)
print('✅ Features ready')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

🔄 Preparing features...


Map:   0%|          | 0/282 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

✅ Features ready


## Step 7 — Text Normalizer, Data Collator and Metrics

The `normalize_text` function is applied to both Whisper's output and the reference
transcripts before computing WER. This ensures the metric is not penalizing
capitalization or punctuation differences — only actual word errors count.

In [11]:
import torch
import evaluate
import re
import string
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# ── Text normalizer ──────────────────────────────────────────────────────────
# Applied to BOTH Whisper output and reference transcripts before WER.
# Whisper often outputs 'The weather is nice.' — we normalize to
# 'the weather is nice' so punctuation/caps don't inflate the error rate.
def normalize_text(text):
    text = text.lower()
    # Remove punctuation except apostrophes (keep contractions)
    punct = string.punctuation.replace("'", "")
    text = text.translate(str.maketrans('', '', punct))
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ── Metrics ───────────────────────────────────────────────────────────────────
metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    # Normalize both sides before scoring
    pred_str  = [normalize_text(p) for p in pred_str]
    label_str = [normalize_text(l) for l in label_str]
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

print('✅ Normalizer, collator and metrics ready')
print(f'\n🔍 Normalizer test:')
print(f'  Input : "The Weather is Nice Today."')
print(f'  Output: "{normalize_text("The Weather is Nice Today.")}"')

✅ Normalizer, collator and metrics ready

🔍 Normalizer test:
  Input : "The Weather is Nice Today."
  Output: "the weather is nice today"


## Step 8 — 🧪 Experiment Configuration

Edit this cell to define your experiments. Each runs a full training loop and results are compared at the end by WER (lower is better).

- **exp1_conservative** — low LR, few epochs, safe baseline
- **exp2_moderate** — balanced
- **exp3_aggressive** — higher risk of overfitting on small datasets

In [12]:
# ✏️ Edit this cell to configure your experiments
EXPERIMENTS = [
    {
        'name': 'exp1_conservative',
        'learning_rate': 1e-5,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 8,
        'warmup_steps': 10,
    },
    {
        'name': 'exp2_moderate',
        'learning_rate': 5e-5,
        'num_train_epochs': 5,
        'per_device_train_batch_size': 8,
        'warmup_steps': 20,
    },
    {
        'name': 'exp3_aggressive',
        'learning_rate': 1e-4,
        'num_train_epochs': 8,
        'per_device_train_batch_size': 8,
        'warmup_steps': 30,
    },
]

print(f'🧪 {len(EXPERIMENTS)} experiments configured:')
for exp in EXPERIMENTS:
    print(f"  → {exp['name']} | LR: {exp['learning_rate']} | Epochs: {exp['num_train_epochs']}")

🧪 3 experiments configured:
  → exp1_conservative | LR: 1e-05 | Epochs: 3
  → exp2_moderate | LR: 5e-05 | Epochs: 5
  → exp3_aggressive | LR: 0.0001 | Epochs: 8


## Step 9 — Run All Experiments

In [14]:
from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import json

all_results = []

for exp in EXPERIMENTS:
    print('\n' + '='*60)
    print(f"🚀 Running: {exp['name']}")
    print(f"   LR: {exp['learning_rate']} | Epochs: {exp['num_train_epochs']}")
    print('='*60)

    model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-tiny.en')
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []

    output_dir = f"/content/whisper-finetuned-{exp['name']}"

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=exp['per_device_train_batch_size'],
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        learning_rate=exp['learning_rate'],
        warmup_steps=exp['warmup_steps'],
        num_train_epochs=exp['num_train_epochs'],
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='wer',
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=225,
        logging_steps=5,
        report_to=['none'],
        fp16=True,
        dataloader_num_workers=2,
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=processor.feature_extractor,
    )

    trainer.train()
    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)

    eval_results = trainer.evaluate()
    wer = eval_results.get('eval_wer', None)

    result = {
        'experiment': exp['name'],
        'learning_rate': exp['learning_rate'],
        'epochs': exp['num_train_epochs'],
        'wer': round(wer, 2) if wer else 'N/A',
        'model_path': output_dir
    }
    all_results.append(result)
    print(f"\n✅ {exp['name']} done | WER: {result['wer']}%")

with open('/content/experiment_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('\n🎉 All experiments complete!')


🚀 Running: exp1_conservative
   LR: 1e-05 | Epochs: 3


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Wer
1,0.776968,0.530622,17.582418
2,0.369784,0.408580,15.506716
3,0.329644,0.397270,15.506716


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ exp3_aggressive done | WER: 14.65%

🎉 All experiments complete!


## Step 10 — Compare Experiment Results

In [15]:
import json

with open('/content/experiment_results.json') as f:
    results = json.load(f)

results_df = pd.DataFrame(results)[['experiment', 'learning_rate', 'epochs', 'wer', 'model_path']]
results_df['wer'] = pd.to_numeric(results_df['wer'], errors='coerce')
results_df = results_df.sort_values('wer')

print('\n📊 Experiment Results (sorted by WER — lower is better):')
print('=' * 60)
print(results_df[['experiment', 'learning_rate', 'epochs', 'wer']].to_string(index=False))
print('=' * 60)

best = results_df.iloc[0]
print(f"\n🏆 Best experiment: {best['experiment']} with WER: {best['wer']}%")


📊 Experiment Results (sorted by WER — lower is better):
       experiment  learning_rate  epochs   wer
    exp2_moderate        0.00005       5 12.33
  exp3_aggressive        0.00010       8 14.65
exp1_conservative        0.00001       3 15.51

🏆 Best experiment: exp2_moderate with WER: 12.33%


## Step 11 — Baseline vs Fine-Tuned: Full WER Comparison

This is the key step. We run **both** the original Whisper Tiny and the best
fine-tuned model on the **same held-out val set** and compare WER side by side.

Both predictions and references are normalized before scoring so the comparison is fair.

- **Significant improvement** = WER dropped by 5%+ → fine-tuning worked well
- **Modest improvement** = WER dropped 1–5% → some benefit, more data would help
- **No improvement / worse** = model overfit or dataset too small — try exp1 conservative

In [16]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import soundfile as sf
import numpy as np
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def run_inference(model, processor, val_df, label='Model'):
    """Run inference on the full val set. Returns (wer, predictions, references)."""
    model.eval()
    model = model.to(device)
    preds = []
    refs  = []
    for i, row in val_df.iterrows():
        audio, _ = sf.read(row['path'])
        audio = audio.astype(np.float32)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        inputs = processor(
            audio, sampling_rate=16000, return_tensors='pt'
        ).input_features.to(device)
        with torch.no_grad():
            generated_ids = model.generate(inputs)
        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        preds.append(normalize_text(pred))
        refs.append(normalize_text(row['transcript']))
        if (len(preds)) % 50 == 0:
            print(f'  [{label}] {len(preds)}/{len(val_df)} done...')
    wer = 100 * metric.compute(predictions=preds, references=refs)
    return wer, preds, refs

# ── Baseline: original Whisper Tiny ──────────────────────────────────────────
print('🔄 Evaluating baseline Whisper Tiny on val set...')
baseline_model     = WhisperForConditionalGeneration.from_pretrained('openai/whisper-tiny.en')
baseline_processor = WhisperProcessor.from_pretrained('openai/whisper-tiny.en')
baseline_wer, baseline_preds, refs = run_inference(
    baseline_model, baseline_processor, val_df.reset_index(), label='Baseline'
)
print(f'\n  Baseline WER: {baseline_wer:.2f}%')

# ── Fine-tuned: best experiment ───────────────────────────────────────────────
best_model_path = results_df.iloc[0]['model_path']
best_exp_name   = results_df.iloc[0]['experiment']
print(f'\n🔄 Evaluating best fine-tuned model ({best_exp_name})...')
finetuned_model     = WhisperForConditionalGeneration.from_pretrained(best_model_path)
finetuned_processor = WhisperProcessor.from_pretrained(best_model_path)
finetuned_wer, finetuned_preds, _ = run_inference(
    finetuned_model, finetuned_processor, val_df.reset_index(), label='Fine-tuned'
)
print(f'\n  Fine-tuned WER: {finetuned_wer:.2f}%')

# ── Summary ───────────────────────────────────────────────────────────────────
improvement = baseline_wer - finetuned_wer
if improvement > 5:
    verdict = '✅ Significant improvement — fine-tuning worked well'
elif improvement > 0:
    verdict = '🟡 Modest improvement — more data would help further'
elif improvement == 0:
    verdict = '➡️  No change'
else:
    verdict = '❌ Performance got worse — model likely overfit, try exp1_conservative'

print('\n' + '='*60)
print('📋 FINAL COMPARISON (val set, normalized WER)')
print('='*60)
print(f'  Val set size          : {len(val_df)} clips')
print(f'  Baseline Whisper Tiny : {baseline_wer:.2f}% WER')
print(f'  Fine-tuned ({best_exp_name[:20]}) : {finetuned_wer:.2f}% WER')
print(f'  Difference            : {improvement:+.2f}% WER')
print(f'  Verdict               : {verdict}')
print('='*60)

# ── Side by side samples ──────────────────────────────────────────────────────
print('\n🔍 Sample predictions (first 5 val clips):')
sample_df = val_df.reset_index().head(5)
for i, row in sample_df.iterrows():
    print(f'\n  REF      : "{normalize_text(row["transcript"])}"')
    print(f'  BASELINE : "{baseline_preds[i]}"')
    print(f'  FINETUNED: "{finetuned_preds[i]}"')

🔄 Evaluating baseline Whisper Tiny on val set...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

  [Baseline] 50/50 done...

  Baseline WER: 18.32%

🔄 Evaluating best fine-tuned model (exp2_moderate)...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

  [Fine-tuned] 50/50 done...

  Fine-tuned WER: 12.45%

📋 FINAL COMPARISON (val set, normalized WER)
  Val set size          : 50 clips
  Baseline Whisper Tiny : 18.32% WER
  Fine-tuned (exp2_moderate) : 12.45% WER
  Difference            : +5.86% WER
  Verdict               : ✅ Significant improvement — fine-tuning worked well

🔍 Sample predictions (first 5 val clips):

  REF      : "a simple definition is machine learning is about building algorithms that learn patterns from data and use those patterns to make predictions"
  BASELINE : "a simple definition is machine learning is about building algorithms that learn patterns from data and use those patterns to make predictions"
  FINETUNED: "a simple definition is machine learning is about building algorithms that learn patterns from data and use those patterns to make predictions"

  REF      : "historical junction its interesting to note fft wasnt invented by one person"
  BASELINE : "historical junction as interesting to note fft w

## Step 12 — Download Your Best Model

In [17]:
import shutil
from google.colab import files

best_path = results_df.iloc[0]['model_path']
zip_path  = '/content/best_model.zip'

shutil.make_archive('/content/best_model', 'zip', best_path)
print(f'📦 Zipping {best_path}...')
print('⬇️  Downloading...')
files.download(zip_path)

📦 Zipping /content/whisper-finetuned-exp2_moderate...
⬇️  Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 📋 Recording Guidelines

### Environment
- Record in a **quiet room** — close windows, turn off fans/AC
- Avoid echo-y rooms — carpeted rooms work better
- Keep a **consistent mic position** every session

### Technical
- iPhone M4A is fine — Step 3 converts everything to WAV automatically
- Aim for **5–20 second clips** per recording
- Any sample rate is fine — Step 3 resamples to 16kHz

### Content
- Read varied sentences — different lengths, topics, speeds
- Speak naturally as you normally would
- Avoid long pauses in the middle of clips

### Sample test sentences:
```
The quick brown fox jumps over the lazy dog.
I went to the store to buy some groceries yesterday.
Can you please pass me the remote control?
The meeting is scheduled for three o'clock tomorrow afternoon.
My favorite food is pizza with extra cheese.
```